# 01 · Pruebas de extracción

Objetivo: comprobar que las fuentes devuelven datos útiles antes de cerrar el alcance del proyecto.

1. **API Overpass (OpenStreetMap):** organizaciones de un municipio de prueba (Oliveira do Hospital).
2. **API del INE:** población por municipio.
3. **Portal BASE (dados.gov.pt):** lista de archivos de contratos públicos.

Las respuestas se guardan en `data/raw/` para no repetir peticiones. Esa carpeta no se sube a GitHub porque contiene contactos.

> **Cómo leer este notebook.** Antes de cada celda de código hay un bloque **Cómo funciona**: qué hace la celda, por qué se hace así y qué herramientas usa. Cuando algo no se ha visto en clase, se marca como **Nuevo** y se explica con más calma. Las funciones que se llaman (`recoleccion.…`, `cleaning.…`, `analisis.…`) están en la carpeta `src/`, con su explicación en el docstring.

**Cómo funciona**

- **Qué hace:** prepara el notebook. Calcula la carpeta raíz del proyecto, la añade a `sys.path` para poder importar los módulos de `src/`, los recarga y lee `config.yaml` en un diccionario.
- **Por qué así:** el código que se repite vive en `src/` y los valores que pueden cambiar (municipios, URLs, rutas, pesos) en `config.yaml`. Así el notebook solo ordena los pasos. `importlib.reload` vuelve a leer `src/` sin reiniciar el kernel cuando se cambia una función.
- **Herramientas:** `import`, diccionarios, `pathlib.Path` (rutas que funcionan igual en Windows y en Mac).
- **Nuevo (no visto en clase):** `importlib.reload`, `sys.path` y `pathlib`.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

# Carpeta raíz del proyecto (este notebook está dentro de notebooks/)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

from src import recoleccion

# Recarga el código de src/ por si ha cambiado (así no hace falta reiniciar el kernel)
importlib.reload(recoleccion)

config = recoleccion.cargar_config(RAIZ / "config.yaml")
RUTA_RAW = RAIZ / config["rutas"]["raw"]
print("Proyecto:", RAIZ)

Proyecto: c:\Users\Asus\Desktop\65_DATA_ANALYTICS_IRONHACK\data-wrangling-project


## 1. Overpass: organizaciones de Oliveira do Hospital

**Cómo funciona**

- **Qué hace:** escribe la consulta para la API Overpass de OpenStreetMap: «dentro del municipio de Oliveira do Hospital, dame las cámaras, clubes, pabellones, colegios, centros comunitarios y bomberos».
- **Por qué así:** la consulta la construye una función a partir de las etiquetas de `config.yaml`, para usar la misma con los 66 municipios. `admin_level=7` es el nivel de municipio en Portugal y `out center tags` pide el centro de cada edificio y sus etiquetas (nombre, email, teléfono…).
- **Herramientas:** funciones, f-strings, `join`.
- **Nuevo (no visto en clase):** el lenguaje de consultas de Overpass (Overpass QL). `nwr` busca nodos, vías y relaciones: puntos, edificios y grupos de edificios.

In [2]:
municipio = config["region"]["municipio_prueba"]
consulta = recoleccion.construir_consulta_overpass(municipio, config["overpass"]["etiquetas"])
print(consulta)

[out:json][timeout:40][maxsize:67108864];
area["name"="Oliveira do Hospital"]["admin_level"="7"]->.municipio;
(
  nwr["amenity"="townhall"](area.municipio);
  nwr["club"="sport"](area.municipio);
  nwr["leisure"="sports_centre"](area.municipio);
  nwr["amenity"="school"](area.municipio);
  nwr["amenity"="community_centre"](area.municipio);
  nwr["amenity"="fire_station"](area.municipio);
);
out center tags;


**Cómo funciona**

- **Qué hace:** si la respuesta ya está guardada en `data/raw/`, la lee del disco; si no, la pide a la API y la guarda en JSON.
- **Por qué así:** es el patrón de caché de todo el proyecto: no se repiten peticiones (los servidores públicos se saturan y pueden bloquearnos) y el análisis se puede repetir sin internet.
- **Herramientas:** `if/else`, `Path.exists()`, `requests` (dentro de `pedir_overpass`), JSON.

In [3]:
nombre_archivo = "overpass_" + municipio.lower().replace(" ", "_") + ".json"
ruta_osm = RUTA_RAW / nombre_archivo

# Si la respuesta ya está guardada, la leemos; si no, la pedimos a la API y la guardamos
if ruta_osm.exists():
    datos_osm = recoleccion.leer_json(ruta_osm)
    print("Leído del archivo guardado:", ruta_osm.name)
else:
    datos_osm = recoleccion.pedir_overpass(consulta, config["overpass"])
    recoleccion.guardar_json(datos_osm, ruta_osm)
    print("Descargado de la API y guardado:", ruta_osm.name)

elementos = datos_osm["elements"]
print("Elementos recibidos:", len(elementos))

Leído del archivo guardado: overpass_oliveira_do_hospital.json
Elementos recibidos: 42


**Cómo funciona**

- **Qué hace:** convierte las etiquetas de cada elemento en una tabla y cuenta cuántos elementos tienen cada etiqueta.
- **Por qué así:** antes de decidir qué columnas usar hay que ver qué trae la fuente. Aquí se ve que el email aparece a veces como `contact:email` y que casi nadie tiene año de fundación.
- **Herramientas:** list comprehension, `pd.DataFrame`, `notna().sum()`, `sort_values`, `head`.

In [4]:
# Una fila por elemento con todas sus etiquetas, para ver cuántas columnas trae OSM
etiquetas_todas = pd.DataFrame([elemento.get("tags", {}) for elemento in elementos])
print("Columnas que trae OSM:", etiquetas_todas.shape[1])

# Cuántos elementos tienen cada etiqueta (las 25 más frecuentes)
etiquetas_todas.notna().sum().sort_values(ascending=False).head(25)

Columnas que trae OSM: 38


amenity            35
name               33
addr:city          17
addr:postcode      16
addr:street        15
operator:type      15
contact:email      15
ref                15
ref:qzp            15
contact:phone      15
grades             14
operator           13
operator:ref       13
building            8
contact:fax         7
leisure             6
contact:website     6
short_name          4
alt_name            4
sport               3
wikidata            3
townhall:type       3
fixme               2
email               2
phone               2
dtype: int64

**Cómo funciona**

- **Qué hace:** recorre los elementos y se queda con las columnas útiles: tipo, nombre, etiquetas de segmento, contactos, año de fundación y coordenadas.
- **Por qué así:** OpenStreetMap guarda el mismo dato con dos nombres (`email` o `contact:email`), por eso `a or b`. Las coordenadas de los edificios vienen en `center`. Solo se muestran columnas sin contactos porque las salidas del notebook se publican en GitHub.
- **Herramientas:** bucle `for`, diccionarios, `.get()` con valor por defecto, `pd.DataFrame`.

In [5]:
filas = []
for elemento in elementos:
    etiquetas = elemento.get("tags", {})
    centro = elemento.get("center", {})  # los edificios traen las coordenadas en "center"
    filas.append({
        "osm_tipo": elemento["type"],
        "osm_id": elemento["id"],
        "nombre": etiquetas.get("name"),
        "amenity": etiquetas.get("amenity"),
        "club": etiquetas.get("club"),
        "leisure": etiquetas.get("leisure"),
        "deporte": etiquetas.get("sport"),
        "web": etiquetas.get("website") or etiquetas.get("contact:website"),
        "email": etiquetas.get("email") or etiquetas.get("contact:email"),
        "telefono": etiquetas.get("phone") or etiquetas.get("contact:phone"),
        "fundacion": etiquetas.get("start_date"),
        "lat": elemento.get("lat", centro.get("lat")),
        "lon": elemento.get("lon", centro.get("lon")),
    })

df_osm = pd.DataFrame(filas)
print(df_osm.shape)
# Solo mostramos columnas sin contactos: las salidas del notebook se suben a GitHub
df_osm[["osm_tipo", "nombre", "amenity", "club", "leisure", "deporte"]].head(15)

(42, 13)


,osm_tipo,nombre,amenity,club,leisure,deporte
0,node,NaN,townhall,NaN,NaN,NaN
1,node,NaN,school,NaN,NaN,NaN
2,node,Junta de Freguesia de Lagares da Beira,townhall,NaN,NaN,NaN
3,node,União das freguesias de Lagos da Beira e Lajeosa,townhall,NaN,NaN,NaN
4,node,Associação Progressiva de Santo António do Alva,NaN,sport,NaN,NaN
5,node,Junta de Freguesia de Avô,townhall,NaN,NaN,NaN
6,node,Jardim de Infância do Largo da Feira,school,NaN,NaN,NaN
7,node,Junta de Freguesia de Alvoco das Várzeas,community_centre,NaN,NaN,NaN
8,node,LAAV - Liga dos Amigos de Alvoco das Várzeas,community_centre,NaN,NaN,NaN
9,way,Câmara Municipal de Oliveira do Hospital,townhall,NaN,NaN,NaN


**Cómo funciona**

- **Qué hace:** asigna a cada organización un segmento comercial (autarquía, deporte, escuela, asociación, bomberos) según sus etiquetas.
- **Por qué así:** es la primera versión, para probar. La definitiva está en `src/cleaning.py` (`asignar_segmento`): allí se separan los clubes de las instalaciones deportivas, porque los pabellones y piscinas suelen ser municipales.
- **Herramientas:** función con `if/elif`, `apply(axis=1)` (fila a fila), `value_counts`.

In [6]:
def asignar_segmento(fila):
    """Devuelve el segmento comercial de la beira. según las etiquetas de OSM."""
    if fila["amenity"] == "townhall":
        return "autarquia"
    elif fila["club"] == "sport" or fila["leisure"] == "sports_centre":
        return "deporte"
    elif fila["amenity"] == "school":
        return "escuela"
    elif fila["amenity"] == "community_centre":
        return "asociacion"
    elif fila["amenity"] == "fire_station":
        return "bomberos"
    return "otro"


df_osm["segmento"] = df_osm.apply(asignar_segmento, axis=1)
df_osm["segmento"].value_counts()

segmento
escuela       21
autarquia      9
deporte        7
asociacion     3
bomberos       2
Name: count, dtype: int64

**Cómo funciona**

- **Qué hace:** mide la calidad: nombres vacíos, nombres repetidos, año de fundación y % de organizaciones con cada contacto.
- **Por qué así:** estas cifras deciden la limpieza: sin nombre no se puede contactar (se eliminan); los repetidos son el mismo sitio dibujado dos veces (punto y edificio); el año de fundación casi no existe, así que los aniversarios no sirven como motivo de contacto.
- **Herramientas:** `isna().sum()`, `duplicated()`, `notna().mean()` (la media de verdaderos y falsos es una proporción), `any(axis=1)`.

In [7]:
# Calidad: nombres vacíos y posibles duplicados
print("Sin nombre:", df_osm["nombre"].isna().sum())
print("Nombres repetidos:", df_osm["nombre"].duplicated().sum())
print("Con año de fundación:", df_osm["fundacion"].notna().sum())

# Cobertura de contactos: % de organizaciones con cada dato
contactos = ["email", "web", "telefono"]
print((df_osm[contactos].notna().mean() * 100).round(1))
print("Con algún contacto:", round(df_osm[contactos].notna().any(axis=1).mean() * 100, 1), "%")

Sin nombre: 9
Nombres repetidos: 9
Con año de fundación: 1
email       40.5
web         14.3
telefono    40.5
dtype: float64
Con algún contacto: 40.5 %


## 2. INE: población por municipio

Primero pedimos los metadatos del indicador para confirmar qué es y qué dimensiones tiene. Después pedimos los datos de un año.

**Cómo funciona**

- **Qué hace:** pide (o lee) los metadatos del indicador de población del INE: nombre, años y dimensiones.
- **Por qué así:** los códigos del INE no dicen nada por sí solos; antes de usar un indicador se comprueba qué mide.
- **Herramientas:** `requests.get` con parámetros (dentro de `pedir_metadatos_ine`), JSON, `isinstance`.

In [8]:
indicador = config["ine"]["indicador_poblacion"]
ruta_meta = RUTA_RAW / f"ine_metadatos_{indicador}.json"

if ruta_meta.exists():
    meta_ine = recoleccion.leer_json(ruta_meta)
else:
    meta_ine = recoleccion.pedir_metadatos_ine(indicador, config["ine"])
    recoleccion.guardar_json(meta_ine, ruta_meta)

# Exploramos la estructura de la respuesta
print(type(meta_ine))
primero_meta = meta_ine[0] if isinstance(meta_ine, list) else meta_ine
print(list(primero_meta.keys()))

<class 'list'>
['IndicadorCod', 'IndicadorNome', 'Periodic', 'PrimeiroPeriodo', 'UltimoPeriodo', 'UnidadeMedida', 'Potencia10', 'PrecisaoDecimal', 'Lingua', 'DataUltimaAtualizacao', 'DataExtracao', 'Dimensoes', 'Sucesso']


**Cómo funciona**

- **Qué hace:** pide los datos de población de 2023 y muestra su estructura: un diccionario `Dados` con una lista de registros por año.
- **Por qué así:** cada registro trae el código y el nombre del territorio (`geocod`, `geodsg`), el sexo (`dim_3`), la edad (`dim_4`) y el valor. Son 19.608 registros porque viene todo el país por sexo y edad: en la limpieza se filtran los municipios y los totales.
- **Herramientas:** diccionarios y listas anidados, `.items()`, slicing (`[:3]`).

In [9]:
periodo = config["ine"]["periodo"]
ruta_pob = RUTA_RAW / f"ine_{indicador}_{periodo}.json"

if ruta_pob.exists():
    datos_pob = recoleccion.leer_json(ruta_pob)
else:
    datos_pob = recoleccion.pedir_datos_ine(indicador, config["ine"], periodo)
    recoleccion.guardar_json(datos_pob, ruta_pob)

primero_pob = datos_pob[0] if isinstance(datos_pob, list) else datos_pob
print(list(primero_pob.keys()))

# Si la respuesta trae datos, vemos cuántos registros hay y los tres primeros
if "Dados" in primero_pob:
    for anio, registros in primero_pob["Dados"].items():
        print(anio, "registros:", len(registros))
        print(registros[:3])

['IndicadorCod', 'IndicadorDsg', 'MetaInfUrl', 'DataExtracao', 'DataUltimoAtualizacao', 'UltimoPref', 'Dados', 'Sucesso']
2023 registros: 19608
[{'geocod': '2004901', 'geodsg': 'Corvo', 'dim_3': '2', 'dim_3_t': 'M', 'dim_4': '91', 'dim_4_t': '80 - 84 anos', 'ind_string': '2', 'valor': '2'}, {'geocod': '2004901', 'geodsg': 'Corvo', 'dim_3': '1', 'dim_3_t': 'H', 'dim_4': '91', 'dim_4_t': '80 - 84 anos', 'ind_string': '3', 'valor': '3'}, {'geocod': '2004901', 'geodsg': 'Corvo', 'dim_3': '1', 'dim_3_t': 'H', 'dim_4': '92', 'dim_4_t': '85 e mais anos', 'ind_string': '3', 'valor': '3'}]


## 3. Portal BASE: archivos disponibles

Antes de descargar, vemos qué archivos tiene el dataset y cuánto pesan.

**Cómo funciona**

- **Qué hace:** pide a la API de dados.gov.pt la lista de archivos del dataset de contratos públicos y la muestra con su tamaño.
- **Por qué así:** los archivos pesan de 16 a 67 MB; antes de descargar se mira qué hay.
- **Herramientas:** `pd.DataFrame` a partir de una lista de diccionarios, `pd.to_numeric(errors="coerce")` (lo que no es número queda vacío), list comprehension.

In [10]:
ruta_info_base = RUTA_RAW / "base_dataset_info.json"

if ruta_info_base.exists():
    info_base = recoleccion.leer_json(ruta_info_base)
else:
    info_base = recoleccion.listar_recursos_dataset(config["base"]["api_dataset"])
    recoleccion.guardar_json(info_base, ruta_info_base)

recursos = pd.DataFrame(info_base["resources"])
if "filesize" in recursos.columns:
    recursos["mb"] = (pd.to_numeric(recursos["filesize"], errors="coerce") / 1_000_000).round(1)

columnas = [c for c in ["title", "format", "mb", "last_modified"] if c in recursos.columns]
pd.set_option("display.max_colwidth", 80)
recursos[columnas]

,title,format,mb,last_modified
0,contratos2026.zip,zip,43.5,2026-10-04T09:04:17.363000+00:00
1,contratos2026.xlsx,xlsx,51.0,2026-10-04T11:41:44.122000+00:00
2,contratos2012.xlsx,xlsx,16.4,2026-10-04T11:40:10.646000+00:00
3,contratos2022.xlsx,xlsx,44.1,2026-10-04T11:41:05.911000+00:00
4,contratos2025.xlsx,xlsx,67.1,2026-10-04T11:41:35.281000+00:00
5,contratos2024.xlsx,xlsx,58.1,2026-10-04T11:41:24.072000+00:00
6,contratos2023.xlsx,xlsx,49.2,2026-10-04T11:41:14.217000+00:00
7,contratos2021.xlsx,xlsx,46.0,2026-10-04T11:40:58.352000+00:00
8,contratos2020.xlsx,xlsx,41.4,2026-10-04T11:40:50.567000+00:00
9,contratos2019.xlsx,xlsx,35.2,2026-10-04T11:40:43.588000+00:00


**Cómo funciona**

- **Qué hace:** descarga los Excel de los años elegidos, pero solo si `DESCARGAR = True`.
- **Por qué así:** el interruptor evita descargas pesadas sin querer al ejecutar todo el notebook; la descarga real se hace en el 02. `descargar_archivo` descarga por partes de 1 MB (`stream=True`) para no cargar el archivo entero en memoria.
- **Herramientas:** `if`, `iterrows`, `str.contains`.

In [11]:
# Descarga: cambiar a True solo después de revisar la lista de arriba
DESCARGAR = False

if DESCARGAR:
    for anio in config["base"]["anos"]:
        seleccion = recursos[recursos["title"].str.contains(str(anio), na=False)]
        for _, fila in seleccion.iterrows():
            nombre = Path(fila["url"]).name
            print("Descargando", nombre)
            recoleccion.descargar_archivo(fila["url"], RUTA_RAW / "base" / nombre)

## Resumen de las pruebas

- **Overpass:** 42 elementos en Oliveira do Hospital; 9 sin nombre y 9 nombres repetidos (hay que limpiarlos); el 40 % tiene email o teléfono y solo 1 tiene año de fundación.
- **INE:** la respuesta trae 19.608 registros de 2023 (todo el país por sexo y edad): en la limpieza se filtran los municipios y los totales.
- **Portal BASE:** un Excel por año, de 16 a 67 MB: se descargan una vez y se filtran al leerlos.

Las tres fuentes sirven, así que se cerró el alcance y se pasó a la recogida de toda la región (notebook 02).